<a href="https://colab.research.google.com/github/beotavalo/LLM-Zoomcamp-2024/blob/main/Elasticsearch_from_Colab_using_Bonsai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Using Elasticsearch from Colab using [Bonsai](https://bonsai.io)

This notebook comes form [this blog post](https://softwaredoug.com/blog/2022/09/11/using-elasticsearch-from-colab.html) and demonstrates how to use a free tier Bonsai Elasticsearch cluster from a Colab notebook.

## Install Elasticsearch Client, get Retrotech Dataset

I have updated to link Bonsai with colab via URL on 7/31/2024. I tested the client and perform some searches.

In [1]:
![ ! -d 'retrotech' ] && git clone https://github.com/ai-powered-search/retrotech.git
! cd retrotech && git pull
! cd retrotech && tar -xvf products.tgz  && tar -xvf signals.tgz

Cloning into 'retrotech'...
remote: Enumerating objects: 41, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 41 (delta 2), reused 6 (delta 1), pack-reused 33
Receiving objects: 100% (41/41), 214.02 MiB | 21.54 MiB/s, done.
Resolving deltas: 100% (6/6), done.
Already up to date.
products.csv
signals.csv


In [2]:
!pip install elasticsearch==7.10.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.3/49.3 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.1/322.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.9/143.9 kB 10.5 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.0.7
    Uninstalling urllib3-2.0.7:
      Successfully uninstalled urllib3-2.0.7


## Paste in your "Full URL" from Bonsai

See how to set this up in [this blog post](https://softwaredoug.com/blog/2022/09/11/using-elasticsearch-from-colab.html)

In [41]:
# Step 2: Import the necessary library
from elasticsearch import Elasticsearch

# Step 3: Connect to the Bonsai Elasticsearch Cluster
# Replace this with your actual Bonsai Elasticsearch URL
es_url = "Bonsai Access URL"

# Initialize the Elasticsearch client
es = Elasticsearch(es_url)

# Test the connection by getting cluster info
try:
    info = es.info()
    print("Cluster info:", info)
except Exception as e:
    print("Error connecting to Elasticsearch:", e)

Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py", line 704, in urlopen
    conn = self._get_conn(timeout=pool_timeout)
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py", line 299, in _get_conn
    return conn or self._new_conn()
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connectionpool.py", line 253, in _new_conn
    conn = self.ConnectionCls(
  File "/usr/local/lib/python3.10/dist-packages/urllib3/connection.py", line 130, in __init__
    _HTTPConnection.__init__(self, *args, **kw)
  File "/usr/lib/python3.10/http/client.py", line 854, in __init__
    self._validate_host(self.host)
  File "/usr/lib/python3.10/http/client.py", line 1236, in _validate_host
    raise InvalidURL(f"URL can't contain control characters. {host!r} "
http.client.InvalidURL: URL can't contain control characters. 'bonsai access url' (found at least ' ')

During handling of the above exception, another exception occ

Error connecting to Elasticsearch: ConnectionError(('Connection aborted.', InvalidURL("URL can't contain control characters. 'bonsai access url' (found at least ' ')"))) caused by: ProtocolError(('Connection aborted.', InvalidURL("URL can't contain control characters. 'bonsai access url' (found at least ' ')")))


In [17]:
es.ping()

True

## Setup Elasticsearch Client

## Index retrotech data (downloaded in first cell)

## Search!

In [28]:
# Read products and signals with pandas dataframe

import pandas as pd
products = pd.read_csv('/content/retrotech/products.csv', on_bad_lines='skip')
signals = pd.read_csv('/content/retrotech/signals.csv')


In [39]:
products.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 47009 entries, 0 to 47008
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   upc                47009 non-null  int64 
 1   name               47009 non-null  object
 2   manufacturer       47009 non-null  object
 3   short_description  47009 non-null  object
 4   long_description   46866 non-null  object
dtypes: int64(1), object(4)
memory usage: 1.8+ MB


In [40]:
signals.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2172605 entries, 0 to 2172604
Data columns (total 5 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   query_id     object
 1   user         object
 2   type         object
 3   target       object
 4   signal_time  object
dtypes: object(5)
memory usage: 82.9+ MB


In [18]:
import csv
from elasticsearch.helpers import bulk
from elasticsearch import RequestError

def retrotech_data():
  with open('retrotech/products.csv') as csv_file:
    products_reader = csv.DictReader(csv_file)
    for row in products_reader:
      yield {
        '_source': row,
        '_index': 'retrotech',
        '_id': row['upc']
      }

try:
  es.indices.create('retrotech')
  bulk(es, retrotech_data())
except RequestError:
  print("Not recreating index that already exists")

In [19]:
hits = es.search(index='retrotech', body={'query': {'match': {'name': 'transformers'}}})
hits = hits['hits']['hits']
for hit in hits:
  print(hit['_source']['name'])


Transformers - DVD
Transformers - Original Soundtrack - CD
The Transformers: The Movie - DVD
Transformers: War for Cybertron - Windows
Transformers: Cybertron Adventures - Nintendo Wii
Nintendo - Transformers 3 Cybertanium Case
Transformers Japanese Collection: Headmasters - DVD
Transformers: The Game - PlayStation 3
Transformers/Transformers: Revenge of the Fallen: Two-Movie Mega Collection [2 Discs] - Widescreen - DVD
Transformers: Season 2, Vol. 1 - DVD


## Cleanup when done...

In [20]:
es.indices.delete('retrotech')

{'acknowledged': True}